# 03 — The kernel trick

*Notebook 3 of 5 · Support Vector Machines*

So far the street has always been **straight**. A straight street cannot separate **curved** data — two
concentric rings are the textbook case, and a line scores no better than a coin flip on them. This
notebook is the chapter's turning point: **lift** the points into a higher-dimensional space where they
*do* become linearly separable, and the boundary, read back in the original plane, is a curve. The
beautiful part — the **kernel trick** — is that we never have to build that space: the optimization
needs the data only through dot products, so we swap the dot product for a **kernel** and get the
curved boundary from the very same maximum-margin machinery.

**Prerequisites**
- NB 1 (the maximum margin) and NB 2 (the soft margin and the cost C).
- Chapter 03, NB 6 — a straight line **underfits** curved truth; this is the door that opens.

**What you'll be able to do**
- Explain why a straight margin cannot separate curved data.
- **Lift** data with a feature map so it becomes linearly separable, and see the curve come back.
- State the **kernel trick**: the optimization needs only dot products, so replace them with a kernel.
- Use the **RBF** and **polynomial** kernels, and say why the kernel (and its degree) must fit the data.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401  (registers the 3-D projection)
from sklearn.datasets import make_circles, make_moons
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

from ml_course import colors, viz

viz.use_course_style()
SEED = 0
np.random.seed(SEED)
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

X, y = make_circles(n_samples=300, factor=0.4, noise=0.10, random_state=SEED)
X = StandardScaler().fit_transform(X)

linear_cv = cross_val_score(SVC(kernel="linear"), X, y, cv=CV).mean()
print(f"make_circles: {len(y)} points, an inner ring (class 1) inside an outer ring (class 0)")
print(f"linear SVM cross-validated accuracy: {linear_cv:.3f}  (about a coin flip)")

## When a straight street cannot do the job

NB 1 and NB 2 found the widest *straight* street, and Chapter 03's straight line already showed it
underfits when the truth is curved. Concentric rings are the sharpest example: an inner disk wrapped by
an outer ring (built here by `make_circles`, a synthetic generator new to us, like `make_moons` earlier). No straight line can keep one class on each side — the linear SVM above confirms it,
scoring barely above chance.

## Lift it: add the coordinate a line is missing

A line fails because it can only read $x_1$ and $x_2$ directly. What separates these rings is something
a line never sees: the **distance from the centre**. So give each point a **new coordinate** built from
the old ones,

$$ r^2 = x_1^2 + x_2^2, $$

small for the inner ring, large for the outer one. This is a **feature map**: lifting each 2-D point up
into a 3-D space $(x_1, x_2, r^2)$.

In [ ]:
r2 = (X**2).sum(axis=1)  # the lift: squared distance from the centre

for k, name in [(1, "inner ring"), (0, "outer ring")]:
    rr = r2[y == k]
    print(f"  class {k} ({name}): r^2 mean {rr.mean():.2f}, range [{rr.min():.2f}, {rr.max():.2f}]")

threshold = 1.5  # any value between the inner max and the outer min separates them
acc = ((r2 <= threshold) == (y == 1)).mean()
print(f"  a single threshold on r^2 at t = {threshold}: accuracy {acc:.3f}")

**Read the result.** One extra coordinate did what no line in the original plane could: the two
classes, hopelessly nested in 2-D, fall on opposite sides of a single cut in $r^2$ (inner ring below
$\approx 1.5$, outer ring above). A straight rule in the **lifted** space is a curved rule back in the
original one.

In [ ]:
fig = plt.figure(figsize=(13, 5.5))

ax2d = fig.add_subplot(1, 2, 1)
for k in (0, 1):
    ax2d.scatter(*X[y == k].T, color=colors.CLASS_CYCLE[k], edgecolor=colors.COLORS["text"],
                 linewidth=0.6, s=35)
ax2d.set_title("In 2-D: no line separates the rings")
ax2d.set_xlabel("feature 1 (standardized)")
ax2d.set_ylabel("feature 2 (standardized)")

ax3d = fig.add_subplot(1, 2, 2, projection="3d")
for k in (0, 1):
    mask = y == k
    ax3d.scatter(X[mask, 0], X[mask, 1], r2[mask], color=colors.CLASS_CYCLE[k],
                 edgecolor=colors.COLORS["text"], linewidth=0.3, s=22)
gx, gy = np.meshgrid(np.linspace(X[:, 0].min(), X[:, 0].max(), 10),
                     np.linspace(X[:, 1].min(), X[:, 1].max(), 10))
ax3d.plot_surface(gx, gy, np.full_like(gx, threshold), alpha=0.25, color=colors.COLORS["muted"])
ax3d.view_init(elev=12, azim=-60)
ax3d.set_title("Lifted to 3-D: a flat plane separates them")
ax3d.set_xlabel("feature 1")
ax3d.set_ylabel("feature 2")
ax3d.set_zlabel("r^2 = x1^2 + x2^2")
fig.tight_layout()
plt.show()

**Read the figure.** On the left, the rings in their original 2-D plane — no straight line works.
On the right, the same points lifted to $(x_1, x_2, r^2)$: the inner ring drops low and the outer ring
rides high, so a **flat plane** (the grey sheet at $r^2 \approx 1.5$) slices cleanly between them. That
plane, projected back down to the 2-D plane, is exactly the **circle** that separates the rings — a
line up there, a curve down here.

## The kernel trick: skip building the lift

Lifting by hand worked, but useful lifts are usually huge — too big to build. Here is the escape. The
SVM's solution depends on the data **only through dot products** between points (NB 1's "Going
further"). So we never form the lift at all: we compute the dot product *in the lifted space* directly,
with a **kernel** $K(x, x')$. The workhorse is the **RBF (Gaussian) kernel**

$$ K(x, x') = \exp\!\big(-\gamma \lVert x - x' \rVert^2\big), $$

whose reach is set by $\gamma$ (we use its default, `gamma='scale'`, here and tune it in NB 4). It
corresponds to a lift we never have to write down.

In [ ]:
rbf = SVC(kernel="rbf").fit(X, y)
rbf_cv = cross_val_score(SVC(kernel="rbf"), X, y, cv=CV).mean()
print(f"RBF SVM cross-validated accuracy: {rbf_cv:.3f}  ({rbf.n_support_.sum()} support vectors)")
print("we never computed r^2 -- the kernel did the lift implicitly")

fig = viz.plot_svm_decision(rbf, X, y)
fig.axes[0].set_title("RBF kernel: a circular boundary, drawn in 2-D")
fig.axes[0].set_xlabel("feature 1 (standardized)")
fig.axes[0].set_ylabel("feature 2 (standardized)")
plt.show()

**Read the figure.** The RBF kernel drew the **same circular street** we built by hand — but in
the original 2-D plane, and **without ever computing** $r^2$. The kernel did the lift implicitly; the
ringed support vectors now sit along a curved margin. That is the whole trick: a curved boundary out of
the very same maximum-margin machinery, by swapping one dot product for a kernel.

## The kernel is a choice — and it must fit the data

RBF is one kernel; the **polynomial** kernel $(\gamma\, x\cdot x' + c)^d$ is another, with the
**degree** $d$ setting how curvy the boundary can be. The kernel's shape has to match the data's — and
the rings make that vivid.

In [ ]:
fig, (ax_deg2, ax_deg3) = plt.subplots(1, 2, figsize=(13, 5.5))
for ax, degree in [(ax_deg2, 2), (ax_deg3, 3)]:
    clf = SVC(kernel="poly", degree=degree).fit(X, y)
    cv = cross_val_score(SVC(kernel="poly", degree=degree), X, y, cv=CV).mean()
    viz.plot_svm_decision(clf, X, y, ax=ax)
    ax.set_title(f"polynomial degree {degree}   (CV accuracy {cv:.3f})")
    ax.set_xlabel("feature 1 (standardized)")
    ax.set_ylabel("feature 2 (standardized)")
fig.tight_layout()
plt.show()

**Read the figure.** Degree 2 nails the rings (CV 1.000) — no surprise, since they *are* a
degree-2 form, $x_1^2 + x_2^2$. The default degree 3 nearly fails (CV about 0.61), barely beating the
linear line: with the default offset $c = 0$, an odd degree builds only odd powers of $x_1, x_2$, so it
cannot form the even radial $x_1^2 + x_2^2$ the rings need. Choosing a kernel — and its degree — is a
real modelling decision, not a default to accept blindly. We tune these knobs honestly in NB 4.

## Beyond circles

The kernel trick is general, not a trick for rings only. The two-moons set that Chapter 03's straight
line could not fit is carved equally cleanly by the same RBF kernel.

In [ ]:
Xm, ym = make_moons(n_samples=300, noise=0.20, random_state=SEED)
Xm = StandardScaler().fit_transform(Xm)
lin_m = cross_val_score(SVC(kernel="linear"), Xm, ym, cv=CV).mean()
rbf_m = cross_val_score(SVC(kernel="rbf"), Xm, ym, cv=CV).mean()
print(f"make_moons: linear CV {lin_m:.3f}  vs  RBF CV {rbf_m:.3f}")

fig = viz.plot_svm_decision(SVC(kernel="rbf").fit(Xm, ym), Xm, ym)
fig.axes[0].set_title("RBF kernel on the two moons")
fig.axes[0].set_xlabel("feature 1 (standardized)")
fig.axes[0].set_ylabel("feature 2 (standardized)")
plt.show()

**Read the figure.** The same RBF kernel bends the boundary around the two interlocking
crescents, lifting accuracy from 0.84 (linear) to 0.97. Whatever the curve, the kernel lets the linear
maximum-margin machinery follow it — circles, moons, or otherwise.

## Honest limits

- The kernel is **implicit**: for the RBF there is no finite feature map to write down (the lift is
  infinite-dimensional) — which is exactly why we let the kernel do it rather than build it.
- The kernel and its knobs **must fit the data**: the degree-3 polynomial nearly failing on the rings
  is the warning. The RBF's $\gamma$ controls under- and over-fitting, and we tune it in NB 4.
- As always, the features must be on a comparable **scale** (we standardized them) — *why* is the
  headline of NB 5.

## Your turn

1. **Easy — why no line works.** In one sentence, explain why no straight line can separate an inner
   disk from an outer ring.
2. **Medium — lift three points.** Compute $r^2 = x_1^2 + x_2^2$ for the points $(0.2, 0.1)$,
   $(1.6, 0.0)$, and $(-1.2, 1.3)$, and say which side of the threshold $t = 1.5$ each lands on.
3. **Harder — the degree matters.** Fit the polynomial kernel at degree 2 and degree 3 on the rings,
   and explain — from the geometry of the data — why the even degree succeeds where the odd one fails.

## What you built

- You saw a straight margin fail on curved data, and **lifted** the rings with $r^2 = x_1^2 + x_2^2$ so
  a flat plane separates them — a line up there, a circle down here.
- You learned the **kernel trick**: because the SVM needs only dot products, you can replace them with a
  **kernel** and get a curved boundary without ever building the lift.
- You used the **RBF kernel** to reach the circular boundary implicitly (no $r^2$ computed), and the
  **polynomial** kernel where the degree had to match the geometry — and saw the same RBF carve the
  two moons.

**Vocabulary:** feature map · the lift · linearly separable in a higher space · the kernel · the RBF
kernel · $\gamma$ · the polynomial kernel and its degree.

## Going further (optional)

The dot-product-only structure (NB 1–2's "Going further") is what licenses the swap: any function that
behaves like a dot product in *some* space is a valid kernel — formally, one whose kernel matrix is
positive semidefinite (**Mercer's condition**). The RBF corresponds to an infinite-dimensional lift,
which is why we never form it. The dials that make a kernel SVM work — `C`, `kernel`, `gamma`, and
`degree` — are the subject of the next notebook.

## References

- Boser BE, Guyon IM, Vapnik VN (1992). *A training algorithm for optimal margin classifiers.*
  Proc. COLT. DOI: [10.1145/130385.130401](https://doi.org/10.1145/130385.130401) — the kernel trick.
- Cortes C, Vapnik V (1995). *Support-vector networks.* Machine Learning 20:273-297.
  DOI: [10.1007/BF00994018](https://doi.org/10.1007/BF00994018)
- Schölkopf B, Smola A (2002). *Learning with Kernels.* MIT Press — kernels and Mercer's condition.
- Hastie T, Tibshirani R, Friedman J (2009). *The Elements of Statistical Learning*, §12.3.
  DOI: [10.1007/978-0-387-84858-7](https://doi.org/10.1007/978-0-387-84858-7)
- James G, Witten D, Hastie T, Tibshirani R (2021). *An Introduction to Statistical Learning*, §9.3.
  DOI: [10.1007/978-1-0716-1418-1](https://doi.org/10.1007/978-1-0716-1418-1)

*Previous: 02 — The soft margin and the cost C · Next: 04 — The estimator and its parameters*